# 3.0 Post-Training Quantization for a Vision-Language Model

In this notebook you will learn how to:

- Apply FP8 (and optionally NVFP4) post-training quantization to a VLM checkpoint.
- Use ModelOpt's existing [`examples/llm_ptq/hf_ptq.py`](../../llm_ptq/hf_ptq.py) flow — no Megatron-Bridge required for this step.

Prerequisite: notebooks [`01_pruning_vlm.ipynb`](01_pruning_vlm.ipynb) and [`02_distillation_vlm.ipynb`](02_distillation_vlm.ipynb) have produced a pruned + distilled VLM. Quantization runs as a separate, final stage on the resulting HF checkpoint.

---
## 3.1 Why quantization is separate from prune/distill

Pruning and distillation reshape the *architecture* (fewer layers, narrower FFN, smaller attention) — they have to happen in BF16/FP16 because the optimizer needs full-precision gradients. Quantization is a one-shot calibration step on the final weights: it converts FP/BF16 tensors into a low-bit format (FP8, NVFP4, INT8, …) without retraining. So the right order is **prune → distill → quantize**.

ModelOpt's PTQ pipeline already understands VLM checkpoints: `hf_ptq.py` calls `is_multimodal_model` to detect the VLM, walks to `model.language_model`, quantizes only the language tower, and writes a unified HF checkpoint with the vision encoder and projector left in BF16. The `--calib_with_images` flag lets calibration see image-text pairs so activation ranges in the projector → LM interface reflect real multimodal traffic.

---
## 3.2 FP8 PTQ

FP8 is the safe default: H100/B200 natively run FP8 matmuls, and the quality drop is usually under 1% on Qwen-VL-class models.

In [ ]:
!nvidia-smi

In [ ]:
DISTILLED_PATH = "/workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-smoke"
EXPORT_PATH    = "/workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-fp8"

!python ../../llm_ptq/hf_ptq.py \
    --pyt_ckpt_path {DISTILLED_PATH} \
    --export_path   {EXPORT_PATH} \
    --qformat fp8 \
    --kv_cache_qformat none \
    --calib_with_images \
    --calib_size 512 \
    --trust_remote_code

Key flags:

- `--qformat fp8` — FP8 weights + activations (E4M3). Use `nvfp4` for Blackwell.
- `--kv_cache_qformat none` — leave KV-cache in BF16. FP8 KV-cache is faster but can hurt longer-context accuracy.
- `--calib_with_images` — VLM-specific: build the calibration set from image-text pairs (uses `nemotron_vlm_dataset_v2`).
- `--calib_size 512` — number of calibration samples. 512 is a good default; more is rarely worth it.

Run `python ../../llm_ptq/hf_ptq.py --help` for the full flag list — including `nvfp4`, `int8_sq`, AWQ, and KV-cache quant variants.

---
## 3.3 (Optional) NVFP4 PTQ

On Blackwell, swap `fp8` for `nvfp4`. The command is otherwise identical:

```bash
python ../../llm_ptq/hf_ptq.py \
    --pyt_ckpt_path /workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-smoke \
    --export_path   /workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-nvfp4 \
    --qformat nvfp4 \
    --kv_cache_qformat nvfp4 \
    --calib_with_images \
    --calib_size 512 \
    --trust_remote_code
```

---
## 3.4 Verify the FP8 VLM still captions

Quick smoke test: load the FP8 export with `AutoModelForImageTextToText` and run the same image-captioning prompt used in notebooks 01 and 02.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from IPython.display import display
from PIL import Image
import requests, torch

EXPORT_PATH = "/workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-fp8"
vlm = AutoModelForImageTextToText.from_pretrained(
    EXPORT_PATH, dtype="auto", device_map="auto", trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(EXPORT_PATH, trust_remote_code=True)

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

preview = image.copy()
preview.thumbnail((512, 512))
display(preview)

messages = [{"role": "user", "content": [
    {"type": "image", "image": image},
    {"type": "text", "text": "Describe the image in one short sentence."},
]}]
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
).to(vlm.device)
out = vlm.generate(**inputs, max_new_tokens=64, do_sample=False)
print("Caption:", processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0])